# Stage 1 — Qwen3-32B value axis (Colab A100)

Extract activations from ICRL JSON through **Qwen/Qwen3-32B** (`enable_thinking=True`), build the value axis, run the held-out AUROC gate.

**Runtime:** A100 80GB (bf16 ~64GB weights).

## Two ICRL upload paths (run **one** upload cell below, not both)

| Path | Upload cell | Source file | When to use |
|------|-------------|-------------|-------------|
| **Faithful (production)** | "Upload — Opus / OpenRouter" | `icrl_32b.json` | Final 32B axis. Generate on laptop with Claude Opus 4.6 via OpenRouter (`--backend openrouter --target-model Qwen3-32B --n 300`). |
| **Proxy smoke test** | "Upload — proxy ICRL" | `icrl_proxy.json` | Pipeline smoke only. Old Qwen-local ICRL (~67 convs). **Not** paper-faithful; gate may fail — OK if extract finishes and axis shape is `(64, 5120)`. |

Cells after upload are shared: extract + gate always use the `ICRL` path set by whichever upload cell you ran.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Need a GPU runtime (A100)'
print(torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

In [ ]:
import os
REPO = '/content/failure_prediction_research'
if not os.path.isdir(REPO):
    # Prefer your fork URL if different:
    !git clone https://github.com/abdelmagid07/failure_prediction_research.git {REPO}
else:
    %cd {REPO}
    !git pull
%cd {REPO}/stage1
!pip install -q -e .
!pip install -q openai python-dotenv

## Choose ICRL source

Run **exactly one** of the next two cells:

1. **Opus / OpenRouter (faithful)** — if you generated `icrl_32b.json` on your laptop with Claude Opus 4.6.
2. **Proxy smoke** — if you only have the old `icrl_proxy.json` (Qwen-local generator) and want to test 32B extract + gate before OpenRouter credits are ready.

Skip the other upload cell entirely.

In [ ]:
# Upload — Opus / OpenRouter (faithful production path)
# Generate on laptop:
#   python -m stage1.icrl_gen.generate --n 300 --backend openrouter \
#     --target-model Qwen3-32B --min-paragraphs 3 --max-paragraphs 8 \
#     --output data/icrl_32b.json --resume
# Then upload the resulting icrl_32b.json here.
# Do NOT run the proxy upload cell below if you use this cell.

from pathlib import Path
from google.colab import files
from stage1.common.paths import data_file

ICRL = data_file('icrl_32b.json')
ICRL.parent.mkdir(parents=True, exist_ok=True)
if not ICRL.exists():
    print('Upload icrl_32b.json (Claude Opus 4.6 via OpenRouter)...')
    up = files.upload()
    src = Path(next(iter(up)))
    src.rename(ICRL)
print('ICRL path:', ICRL)
print('exists:', ICRL.exists())
if ICRL.exists():
    print('size bytes:', ICRL.stat().st_size)

In [ ]:
# Upload — proxy ICRL (smoke test only)
# Use old icrl_proxy.json (Qwen-local generator, ~67 convs from Downloads).
# Purpose: verify Qwen3-32B load + extract + gate wiring before faithful Opus data exists.
# Not paper-faithful — treat output as a smoke axis, not for Stage-2 science.
# Do NOT run the Opus upload cell above if you use this cell.

from pathlib import Path
from google.colab import files
from stage1.common.paths import data_file

ICRL = data_file('icrl_proxy.json')
ICRL.parent.mkdir(parents=True, exist_ok=True)
if not ICRL.exists():
    print('Upload icrl_proxy.json (proxy / Qwen-local dataset)...')
    up = files.upload()
    src = Path(next(iter(up)))
    src.rename(ICRL)
print('ICRL path:', ICRL)
print('exists:', ICRL.exists())
if ICRL.exists():
    print('size bytes:', ICRL.stat().st_size)

In [ ]:
# Extract + gate (shared — works with either upload cell above)
# Uses whatever path is in ICRL (icrl_32b.json faithful OR icrl_proxy.json smoke).
# Preset qwen32b: Qwen/Qwen3-32B, enable_thinking=True, outputs value_axis_32b.npy
import subprocess, sys

print('Running extract + gate on:', ICRL)
subprocess.run([
    sys.executable, '-m', 'stage1.pipeline.run_gate',
    '--preset', 'qwen32b',
    '--icrl', str(ICRL),
    '--force-extract',
], check=False)  # check=False: download artifacts even if AUROC gate fails (common for proxy smoke)

In [ ]:
# Download artifacts (shared)
# Faithful run: keep as value_axis_32b.npy for Stage 2.
# Proxy smoke: rename locally to e.g. value_axis_32b_smoke.npy so you do not overwrite a later faithful axis.

from google.colab import files
from stage1.common.paths import data_file
import json

manifest = data_file('axis_manifest_32b.json')
if manifest.exists():
    m = json.loads(manifest.read_text())
    print('gate_passed:', m.get('gate_passed'))
    print('primary_layer:', m.get('primary_layer'))
    print('icrl_path:', m.get('icrl_path'))
    print(json.dumps(m, indent=2)[:2000])

for name in [
    'value_axis_32b.npy',
    'axis_manifest_32b.json',
    'auroc_by_layer_32b.json',
    'auroc_by_layer_32b.png',
]:
    p = data_file(name)
    if p.exists():
        files.download(str(p))
    else:
        print('missing:', p)